# Общий эффект добавления номера договора в ключ

Тетрадка считает:
- сколько уникальных карточек получается при базовом ключе `inn + fp_num + fp_type + IDENTIFICATION_DATE`;
- сколько уникальных карточек получится при добавлении `agreement_num` в ключ;
- разницу в абсолюте и процентах;
- среднее и медиану количества договоров на один ИНН.

Экспорт в Excel не выполняется (по умолчанию только вывод в тетрадке).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

DATA_DIR = Path('/home/jovyan/documents/DMUKP_FP_DEF/data')
CRM_FILE = DATA_DIR / 'crm_last_version.csv'

DATE_FROM = pd.Timestamp('2023-01-01')
DATE_TO = pd.Timestamp('2025-12-31')

ALLOWED_SOURCES = ['Н2.0', 'Справочник CRM-системы', 'CRM-система']
SEGMENT_MAP = {
    'ДМСБ (микро)': 'МкБ',
    'ДМСБ (малый)': 'МСБ',
    'ДМСБ (средний)': 'МСБ',
    'ДКБ': 'ККБ',
}

BASE_KEY = ['inn', 'fp_num', 'fp_type', 'IDENTIFICATION_DATE']
KEY_WITH_AGREEMENT = BASE_KEY + ['agreement_num']


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith('.0'):
        s = s[:-2]
    s = s.lstrip('0') or '0'
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print('CRM_FILE:', CRM_FILE)

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

if 'VAL' in raw.columns:
    raw = raw[raw['VAL'].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

raw['inn'] = raw['X_INN'].apply(normalize_inn)
raw['dt'] = pd.to_datetime(raw['IDENTIFICATION_DATE'], dayfirst=True, errors='coerce')
raw = raw[(raw['dt'] >= DATE_FROM) & (raw['dt'] <= DATE_TO)].copy()

raw['segment'] = raw['X_AREA_RESP'].astype(str).str.strip().map(SEGMENT_MAP)
raw = raw[raw['segment'].notna()].copy()

raw = raw.dropna(subset=['inn', 'NUMBER_FP_SFP']).copy()
raw['fp_num'] = raw['NUMBER_FP_SFP'].astype(str).str.strip()
raw['fp_type'] = raw['TYPE_FP'].astype(str).str.strip().replace({'FP': 'ФП', 'SFP': 'СФП'})
raw['agreement_num'] = raw['AGREEMENT_NUM'].astype(str).str.strip()
raw.loc[raw['agreement_num'] == '', 'agreement_num'] = '[EMPTY_AGREEMENT]'

print('Rows after filters:', len(raw))
print('Unique INN:', raw['inn'].nunique())
print('Rows with empty agreement replaced:', int((raw['agreement_num'] == '[EMPTY_AGREEMENT]').sum()))

In [ ]:
cards_base_key = raw.drop_duplicates(BASE_KEY).shape[0]
cards_key_with_agreement = raw.drop_duplicates(KEY_WITH_AGREEMENT).shape[0]
delta_cards = cards_key_with_agreement - cards_base_key
delta_pct_vs_base = round((delta_cards / cards_base_key * 100), 4) if cards_base_key else None

summary_key_compare = pd.DataFrame([
    {
        'cards_base_key': int(cards_base_key),
        'cards_key_with_agreement': int(cards_key_with_agreement),
        'delta_cards': int(delta_cards),
        'delta_pct_vs_base': delta_pct_vs_base,
    }
])

display(summary_key_compare)

In [ ]:
agreements_per_inn = (
    raw.groupby('inn', as_index=False)
    .agg(agreements_per_inn=('agreement_num', pd.Series.nunique))
)

agreements_distribution_summary = pd.DataFrame([
    {
        'inn_count': int(len(agreements_per_inn)),
        'mean_agreements_per_inn': round(float(agreements_per_inn['agreements_per_inn'].mean()), 4) if len(agreements_per_inn) else None,
        'median_agreements_per_inn': float(agreements_per_inn['agreements_per_inn'].median()) if len(agreements_per_inn) else None,
        'p75': float(agreements_per_inn['agreements_per_inn'].quantile(0.75)) if len(agreements_per_inn) else None,
        'p90': float(agreements_per_inn['agreements_per_inn'].quantile(0.90)) if len(agreements_per_inn) else None,
        'max': int(agreements_per_inn['agreements_per_inn'].max()) if len(agreements_per_inn) else None,
    }
])

agreements_per_inn_value_counts = (
    agreements_per_inn['agreements_per_inn']
    .value_counts(dropna=False)
    .rename_axis('agreements_per_inn')
    .reset_index(name='inn_count')
    .sort_values('agreements_per_inn')
    .reset_index(drop=True)
)

print('Статистика количества договоров на ИНН:')
display(agreements_distribution_summary)
print('Распределение agreements_per_inn:')
display(agreements_per_inn_value_counts.head(50))
print('TOP ИНН по количеству договоров:')
display(agreements_per_inn.sort_values('agreements_per_inn', ascending=False).head(30))

In [ ]:
checks = pd.DataFrame([
    {
        'check_name': 'cards_key_with_agreement_ge_cards_base_key',
        'passed': bool(cards_key_with_agreement >= cards_base_key),
    },
    {
        'check_name': 'all_agreements_per_inn_ge_1',
        'passed': bool((agreements_per_inn['agreements_per_inn'] >= 1).all()) if len(agreements_per_inn) else True,
    },
])

rows_before = int(len(raw))
rows_after_base = int(raw.drop_duplicates(BASE_KEY).shape[0])
rows_after_agreement = int(raw.drop_duplicates(KEY_WITH_AGREEMENT).shape[0])

control_totals = pd.DataFrame([
    {'metric': 'rows_before', 'value': rows_before},
    {'metric': 'rows_after_base_key', 'value': rows_after_base},
    {'metric': 'rows_after_key_with_agreement', 'value': rows_after_agreement},
    {'metric': 'removed_by_base_key', 'value': rows_before - rows_after_base},
    {'metric': 'removed_by_key_with_agreement', 'value': rows_before - rows_after_agreement},
])

print('Контрольные проверки:')
display(checks)
print('Контрольные итоги:')
display(control_totals)